# GPU 내구성 진단 노트북 — "이 PC가 부하를 버티는가"

## 이 노트북을 왜 만들었나

측정에 쓴 PC(<호스트명>)가 학습 부하 중 **전원이 통째로 두 번 꺼졌다.** 한 번은
crop 학습(VRAM 여유가 거의 없던 설정)에서, 한 번은 VRAM 여유가 있던 baseline 설정을
오래(12 step, 30분 이상) 돌리는 도중이었다. **이 PC 개별 문제인지 교육장 PC 공통
사양 문제인지 아직 모른다.** `vram_shutdown_probe.ipynb`(기존 측정 노트북)와 이
노트북은 별개다 — 저건 "숫자를 잰다"가 목적이고, 이건 **"버티는가"만** 본다.

## 파일 하나로 바로 돈다

**Stage 0·1(그리고 1-S)은 이 `.ipynb` 파일 하나만 있으면 된다.** 저장소를 clone할
필요도, `src/` 폴더가 옆에 있을 필요도 없다 — 필요한 코드를 전부 셀 안에 그대로
넣었다(다른 노트북·저장소 코드와 중복되지만 의도적이다: 팀원이 파일 하나 받아서
바로 실행할 수 있어야 한다). **Stage 2·3은 실제 9B 모델을 쓰므로 저장소가
필요하다** — 해당 위치에 clone·설치 안내를 마크다운으로 넣었다.

## ⚠️ 먼저 읽을 것

- **실행 중 PC가 갑자기 꺼질 수 있습니다.** 저장 안 된 다른 작업을 먼저 닫아주세요.
- **Stage 0·1(+1-S)만 돌려도 큰 도움이 됩니다.** 9B 모델도, 대회 데이터도, 저장소도
  필요 없다. **총 소요 시간 약 20분**(램프 5분 + 15분 연속 유지). 시간이 없으면
  여기까지만 돌리고 그 결과만 보내줘도 된다.
- Stage 2(추론) / Stage 3(학습)로 갈수록 무거워진다. **아래로 갈수록 부하가 세진다 —
  일부러 순서를 이렇게 짰다.** 어느 단계에서 꺼지는지 자체가 정보다.
- 학습 첫 step에서 1~2분 정지한 것처럼 보일 수 있다(Triton 커널 최초 컴파일 —
  정상이다). **멈춘 줄 알고 Ctrl+C 누르지 말 것**(단, Stage 1-S의 안내가 있는
  대기 구간은 예외 — 거기는 눌러서 중단해도 된다. 셀 안에 표시해뒀다).

## 실행 방법

1. Jupyter가 있는 아무 Python 환경(가급적 `baseline`)에서 이 `.ipynb`만 열어서
   위에서 아래로 순서대로 실행한다. Stage 0·1(+1-S)은 `torch`만 있으면 된다.
2. 직전에 이 노트북을 돌리다 꺼졌었다면, Stage 0 다음 셀이 그 사실을 알려주고
   이어서 할지 처음부터 할지 고를 수 있게 한다.
3. Stage 2·3으로 넘어가려면 그 앞의 안내를 따라 저장소를 먼저 준비한다.
4. 끝나면(또는 셧다운으로 중간에 멈췄으면) 다음 3개 파일을 보내달라:
   - `results_<hostname>.csv` — 단계별 측정 결과
   - `gpulog_<hostname>.csv` — 1초 간격 GPU 전력·온도·VRAM (죽기 직전까지, **어느
     Stage에서 꺼졌는지 컬럼으로 남는다**)
   - `progress_<hostname>.json` — 마지막으로 완료된 단계


## Stage 0 — 환경 정보 (부하 없음, 즉시 끝남, 저장소 불필요)

In [ ]:
import datetime
import json
import os
import platform
import socket
import subprocess
import sys
import threading
import time
from pathlib import Path

HOSTNAME = socket.gethostname()
START_TS = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

print("hostname:", HOSTNAME)
print("start_ts:", START_TS)
print("platform:", platform.platform())
print("processor:", platform.processor())

def nvidia_smi_query(fields):
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True, timeout=10,
    ).stdout.strip()
    return [line.split(", ") for line in out.splitlines()]

print("\n--- GPU / 드라이버 (baseline, 우리 프로세스 시작 전) ---")
for row in nvidia_smi_query("name,driver_version,memory.total,memory.used,memory.free,power.draw,temperature.gpu"):
    GPU_NAME = row[0]
    print("name:", row[0])
    print("driver_version:", row[1])
    print("memory.total (MiB):", row[2])
    print("memory.used (MiB) — 다른 프로그램이 이미 쓰고 있는 양:", row[3])
    print("memory.free (MiB):", row[4])
    print("power.draw (W):", row[5])
    print("temperature.gpu (C):", row[6])

print("\n--- CPU / 시스템 RAM ---")
try:
    import psutil
    print("logical CPUs:", psutil.cpu_count())
    print("system RAM total (GB):", round(psutil.virtual_memory().total / 1e9, 1))
except ImportError:
    print("psutil 없음 — 건너뜀 (pip install psutil로 설치 가능, 필수 아님)")

print("\n--- 전원 관리 (Windows, 얻을 수 있으면만) ---")
if platform.system() == "Windows":
    try:
        out = subprocess.run(["powercfg", "/getactivescheme"], capture_output=True, text=True, timeout=10).stdout
        print("전원 관리 계획:", out.strip())
    except Exception as exc:
        print("powercfg 조회 실패(건너뜀):", exc)
else:
    print("Windows 아님 — 건너뜀")

print("\n--- 패키지 버전 (설치돼 있으면만, Stage 0·1엔 torch만 있으면 된다) ---")
for pkg in ["torch", "transformers", "accelerate", "peft", "bitsandbytes", "fla", "triton"]:
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"{pkg}: 설치 안 됨")


## 진행 상태 파일 — 이어하기

이전에 이 노트북을 돌리다 꺼졌으면 `progress_<hostname>.json`에 마지막으로 완료된
단계가 남아 있다. 그걸 읽어서 알려주고, 어디서부터 시작할지 정한다.


In [ ]:
PROGRESS_PATH = f"progress_{HOSTNAME}.json"
RESULTS_PATH = f"results_{HOSTNAME}.csv"
GPULOG_PATH = f"gpulog_{HOSTNAME}_{START_TS}.csv"

RESULTS_FIELDS = [
    "hostname", "stage", "mode", "long_side", "crops", "crop_long_side", "visual_tokens",
    "peak_vram_torch", "peak_vram_smi", "shared_mem_peak", "max_power_w", "max_temp_c",
    "sec_per_step_steady", "sec_per_step_warmup", "fla_version", "triton_version",
    "kernel_impl", "status",
]

def load_progress() -> dict:
    if Path(PROGRESS_PATH).exists():
        try:
            return json.loads(Path(PROGRESS_PATH).read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}

def mark_progress(stage: str, state: str) -> None:
    """Stage 시작/완료를 그때그때 디스크에 쓴다 — 전원이 끊겨도 마지막 상태가 남는다."""
    prog = load_progress()
    prog[stage] = {"state": state, "ts": datetime.datetime.now().isoformat()}
    prog["_last_updated"] = stage
    Path(PROGRESS_PATH).write_text(json.dumps(prog, ensure_ascii=False, indent=2), encoding="utf-8")

_prev = load_progress()
if _prev:
    print("직전 실행 기록을 찾았다:")
    for k, v in _prev.items():
        if k.startswith("_"):
            continue
        print(f"  {k}: {v['state']} ({v['ts']})")
    print(f"\n마지막으로 다룬 단계: {_prev.get('_last_updated')}")
    print("""
이어서 하려면 아래 RESUME_FROM을 완료된 단계 다음 것으로 바꾸고,
처음부터 다시 하려면 그대로(0) 둔다. 이 셀 실행 후 다음 셀로 넘어간다.
""")
else:
    print("직전 실행 기록 없음 — 처음부터 시작한다.")

# 필요하면 이 값을 0(처음부터) / 1(Stage 1부터) / 2(Stage 2부터) / 3(Stage 3부터)로 바꾼 뒤
# 이 셀을 다시 실행하지 말고 바로 아래로 진행한다.
RESUME_FROM = 0
print("RESUME_FROM =", RESUME_FROM)


## 전력/온도 로거 — 별도 스레드, 1초 간격, **매 줄 flush+fsync** (저장소 불필요)

전원이 끊겨도 이 CSV의 마지막 줄까지는 디스크에 남는다. **현재 Stage 번호를
컬럼으로 같이 남긴다** — 어느 단계에서 꺼졌는지가 이 노트북의 핵심 정보다.
Windows면 WDDM sysmem fallback(시스템 RAM 폴백) 사용량도 같이 잰다. 이 노트북
파일 하나만으로 돌아가야 하므로 필요한 코드를 전부 이 셀 안에 그대로 넣었다
(`src/resources.py`에도 같은 함수가 있다 — 의도된 중복이다).


In [ ]:
import csv

CURRENT_STAGE = {"value": "0-env"}

def set_stage(name: str) -> None:
    CURRENT_STAGE["value"] = name
    print(f"[Stage] {name}")


def _windows_gpu_shared_usage_gb(pid: int) -> float | None:
    """WDDM sysmem fallback으로 시스템 RAM에 흘려보낸 GPU 메모리량(GB), Windows 전용.

    torch/nvidia-smi 둘 다 dedicated VRAM만 본다 — 이 프로세스의 GPU shared 메모리
    사용량은 Windows 성능 카운터로만 잡힌다. 09/17 실측(다른 세션)으로 실제 학습
    중 dedicated 15.67GB + shared 7.04GB가 동시에 잡힌 것을 이 방법으로 확인했다.
    """
    if platform.system() != "Windows":
        return None
    try:
        script = (
            f"(Get-Counter '\\GPU Process Memory(*)\\Shared Usage' -ErrorAction SilentlyContinue)."
            f"CounterSamples | Where-Object {{ $_.Path -like '*pid_{pid}_*' }} | "
            f"Measure-Object -Property CookedValue -Sum | Select-Object -ExpandProperty Sum"
        )
        res = subprocess.run(
            ["powershell", "-NoProfile", "-Command", script],
            capture_output=True, text=True, timeout=15,
        )
        val = res.stdout.strip()
        return float(val) / 1e9 if val else 0.0
    except (OSError, subprocess.SubprocessError, ValueError):
        return None


class NvidiaSmiPeakSampler:
    """with 블록 동안 별도 스레드로 nvidia-smi memory.used peak를 잰다."""

    def __init__(self, interval: float = 0.5):
        self.interval = interval
        self.peak: float | None = None
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _run(self) -> None:
        while not self._stop.is_set():
            try:
                rows = nvidia_smi_query("memory.used")
                v = float(rows[0][0]) / 1024
                self.peak = v if self.peak is None else max(self.peak, v)
            except Exception:
                pass
            self._stop.wait(self.interval)

    def __enter__(self):
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join(timeout=2)


class WindowsSharedMemPeakSampler:
    """with 블록 동안 이 프로세스의 WDDM shared 메모리 peak를 잰다(Windows 전용)."""

    def __init__(self, interval: float = 2.0, pid: int | None = None):
        self.interval = interval
        self.pid = pid or os.getpid()
        self.peak: float | None = None
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _run(self) -> None:
        while not self._stop.is_set():
            v = _windows_gpu_shared_usage_gb(self.pid)
            if v is not None:
                self.peak = v if self.peak is None else max(self.peak, v)
            self._stop.wait(self.interval)

    def __enter__(self):
        if platform.system() == "Windows":
            self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        if self._thread.is_alive():
            self._thread.join(timeout=3)


GPULOG_FIELDS = ["timestamp", "stage", "power_draw_w", "temperature_c",
                  "utilization_pct", "memory_used_mib", "shared_mem_gb_windows"]

_gpulog_stop = threading.Event()

def _gpulog_loop():
    with open(GPULOG_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if f.tell() == 0:
            writer.writerow(GPULOG_FIELDS)
            f.flush()
            os.fsync(f.fileno())
        while not _gpulog_stop.is_set():
            try:
                rows = nvidia_smi_query("power.draw,temperature.gpu,utilization.gpu,memory.used")
                shared = _windows_gpu_shared_usage_gb(os.getpid())
                ts = datetime.datetime.now().isoformat()
                for row in rows:
                    writer.writerow([ts, CURRENT_STAGE["value"], *row, shared if shared is not None else ""])
                f.flush()
                os.fsync(f.fileno())
            except Exception as exc:
                print("gpu log 폴링 실패(계속 시도함):", exc)
            _gpulog_stop.wait(1.0)

_gpulog_thread = threading.Thread(target=_gpulog_loop, daemon=True)
_gpulog_thread.start()
print(f"GPU 로거 시작: {GPULOG_PATH} (1초 간격, 매 줄 flush+fsync, Stage 컬럼 포함)")

def _write_result_row(row: dict) -> None:
    row.setdefault("hostname", HOSTNAME)
    is_new = not Path(RESULTS_PATH).exists()
    with open(RESULTS_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_FIELDS)
        if is_new:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in RESULTS_FIELDS})
        f.flush()
        os.fsync(f.fileno())

print(f"결과 CSV: {RESULTS_PATH}")
mark_progress("0-env", "done")


## Stage 1 — 모델 없는 GPU 부하 램프 + 지속 유지 ★ 가장 중요 (저장소 불필요)

**9B 가중치를 받지 않는다.** 순수 torch 행렬곱만으로 두 단계를 본다:

1. **램프(1-ramp)** — VRAM 점유를 2→4→8→12→14GB로 올리며 각 단계 60초씩 100% 부하
   (약 5분). 어느 점유량부터 이상 징후(전력 급변, 온도 급상승 등)가 나오는지 본다.
2. **지속 유지(1-S)** — 램프 최고 단계(14GB)로 **15분 연속** 100% 부하를 준다.
   **측정에 쓴 PC의 실제 셧다운은 30분 넘게 부하가 걸린 뒤 났다 — 램프만으로는
   (총 5분) 지속 시간이 너무 짧아 재현이 안 될 수 있다.** 이 구간이 모델 없이도
   실제 학습의 "오래 100%를 문다"는 조건만 흉내낸다.

**이 두 단계만으로 셧다운이 재현되면 그것으로 결론이 난다** — 우리 코드·모델과
무관하게 이 PC(또는 이 카드 모델 전체)가 지속 부하를 못 버틴다는 뜻이다.
**총 소요 시간 약 20분**(램프 5분 + 지속 유지 15분). 4대에서 20분이면 끝난다.

지속 유지 구간은 1분마다 남은 시간을 출력한다. **중간에 그만하고 싶으면 이
구간에서는 Ctrl+C(또는 Jupyter의 인터럽트 버튼)를 눌러도 된다** — 결과는 그때까지
기록된 대로 저장된다.

각 단계가 끝날 때마다 결과를 바로 쓰고 다음 단계로 넘어간다 — 한 단계가 죽어도
그 전까지 기록은 남는다.


In [ ]:
import torch

def gpu_load_stage(target_gb: float, hold_seconds: int = 60, stage_name: str | None = None):
    stage_name = stage_name or f"1-ramp-{target_gb}GB"
    set_stage(stage_name)
    row = {"stage": stage_name, "mode": "load_ramp"}
    try:
        # a, b, c(=a@b) 세 개의 (n,n) float32 행렬로 target_gb에 근접시킨다. 정밀하지
        # 않아도 된다 — 목적은 "이 근방 VRAM을 오래 100%로 물렸을 때 버티는가"다.
        n = max(256, int((target_gb * 1e9 / (3 * 4)) ** 0.5))
        a = torch.randn(n, n, device="cuda", dtype=torch.float32)
        b = torch.randn(n, n, device="cuda", dtype=torch.float32)
        # c를 loop 밖에서 한 번만 할당하고 matmul(out=c)로 제자리에 덮어쓴다.
        # `c = a @ b`를 loop 안에서 반복하면 파이썬이 RHS를 먼저 평가해 새 텐서를
        # 만든 "다음에" 옛 c의 참조를 끊기 때문에, 그 찰나 a+b+c(old)+c(new) 네 개가
        # 동시에 올라가 peak이 의도한 3개(a+b+c) 분량의 4/3배로 잡힌다 — 09/18 실측으로
        # 확인(2/4/8/12GB 전부 정확히 4/3배 초과, 14GB에서 그 초과분 때문에 OOM).
        c = torch.empty((n, n), device="cuda", dtype=torch.float32)

        torch.cuda.reset_peak_memory_stats()
        t_end = time.time() + hold_seconds
        iters = 0
        last_report = time.time()
        with NvidiaSmiPeakSampler() as smi, WindowsSharedMemPeakSampler() as shared:
            while time.time() < t_end:
                torch.matmul(a, b, out=c)
                torch.cuda.synchronize()
                iters += 1
                if hold_seconds >= 120 and time.time() - last_report >= 60:
                    remaining_min = (t_end - time.time()) / 60
                    print(f"    ...{stage_name} 진행 중, 남은 시간 약 {remaining_min:.1f}분 "
                          f"(중단하려면 Ctrl+C)")
                    last_report = time.time()
            del c

        row["peak_vram_torch"] = round(torch.cuda.max_memory_allocated() / 1e9, 3)
        row["peak_vram_smi"] = round(smi.peak, 3) if smi.peak is not None else None
        row["shared_mem_peak"] = round(shared.peak, 3) if shared.peak is not None else None
        row["status"] = "ok"
        print(f"  {stage_name}: {iters}회 반복, peak_vram_torch={row['peak_vram_torch']}GB, "
              f"shared_mem_peak={row['shared_mem_peak']}")

        del a, b
        torch.cuda.empty_cache()
    except torch.cuda.OutOfMemoryError as exc:
        row["status"] = "OOM"
        print(f"  {stage_name}: OOM — {exc}")
        torch.cuda.empty_cache()
    except KeyboardInterrupt:
        row["status"] = "interrupted_by_user"
        print(f"  {stage_name}: 사용자가 중단함 — 여기까지 기록하고 넘어간다.")
        try:
            del a, b
        except NameError:
            pass
        torch.cuda.empty_cache()
    except Exception as exc:
        row["status"] = f"ERROR({type(exc).__name__})"
        print(f"  {stage_name}: ERROR — {exc}")
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
    _write_result_row(row)
    mark_progress(stage_name, "done")
    return row


if RESUME_FROM <= 1:
    mark_progress("1-ramp", "start")
    for gb in [2, 4, 8, 12, 14]:
        gpu_load_stage(gb, hold_seconds=60)
    mark_progress("1-ramp", "done")

    mark_progress("1-S-sustained", "start")
    print("\n=== Stage 1-S: 14GB로 15분 연속 유지 시작 (필요하면 Ctrl+C로 중단 가능) ===")
    gpu_load_stage(14, hold_seconds=15 * 60, stage_name="1-S-sustained-14GB-15min")
    mark_progress("1-S-sustained", "done")

    print("\nStage 1(+1-S) 끝. 여기까지만 돌려도 의미 있는 결과다 — 여기서 멈춰도 된다.")
else:
    print("RESUME_FROM 설정에 따라 Stage 1을 건너뛴다.")


---
## ⬇️ 여기부터는 저장소가 필요합니다 (Stage 2·3)

Stage 0·1은 이 `.ipynb` 파일 하나로 끝났다. Stage 2(추론)·3(학습)은 실제 9B 모델을
쓰므로 저장소를 clone하고 의존성을 설치해야 한다.

### 1. 저장소 clone + 브랜치 체크아웃

```bash
git clone <저장소 URL> AI_2_CHALLENGE
cd AI_2_CHALLENGE
git checkout chore/restructure
```

### 2. 가상환경 + 의존성 설치

```bash
python -m venv baseline && baseline\Scripts\activate    # Windows
pip install -e .
```

`pyproject.toml`에 `transformers`가 PyPI 최신 릴리스가 아니라 **git 커밋 고정**으로
들어있다(Qwen3.5 아키텍처가 아직 정식 릴리스에 없어서다 — `SETUP_QWEN35.md` 참조).
`pip install -e .`가 그 커밋을 자동으로 받으니 별도 조치는 필요 없지만, 최초 설치
시 git 클론 + wheel 빌드로 1~2분 더 걸린다.

### 3. ⚠️ 9B 가중치 다운로드 — 용량 크고 시간 걸림

Stage 2를 실행하면 `Qwen/Qwen3.5-9B`(약 19GB)를 처음 한 번 내려받는다. **디스크
여유 공간과 네트워크 상태를 먼저 확인해라.** `HF_HOME` 환경변수로 받을 드라이브를
지정할 수 있다(기본은 `C:\Users\<사용자>\.cache\huggingface`).

### 4. 이 노트북에서 REPO_ROOT 지정

아래 셀이 현재 작업 디렉터리부터 위로 올라가며 `pyproject.toml`을 찾는다. Jupyter를
저장소 밖에서 띄웠다면 이 노트북 파일을 저장소 아래로 옮기거나, `REPO_ROOT`를
직접 지정해라.


In [ ]:
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if not (REPO_ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "저장소를 찾지 못했다. 이 노트북을 AI_2_CHALLENGE 저장소 안(또는 그 하위)으로 "
        "옮기거나, 위 REPO_ROOT를 Path('C:/.../AI_2_CHALLENGE')처럼 직접 지정해라."
    )
print("REPO_ROOT:", REPO_ROOT)


## Stage 2 — 모델 로드 + 추론 (가중치 다운로드 필요, 시간이 좀 걸림)

Full 1024 → 1536 → 2048 순서(가벼운 것부터). `python -m src.run smoke --synthetic`를
서브프로세스로 돌린다 — 설정 사이에 프로세스가 새로 뜨기 때문에 이전 설정의 VRAM
파편화가 다음 설정에 영향을 주지 않는다.

In [ ]:
def _run_subprocess(args, timeout_sec=900):
    proc = subprocess.run(
        [sys.executable, "-m", "src.run", *args],
        cwd=str(REPO_ROOT), capture_output=True, text=True, timeout=timeout_sec,
    )
    return proc.returncode, (proc.stdout or "") + (proc.stderr or "")


def _classify_status(returncode: int, output: str) -> str:
    if returncode == 0:
        return "ok"
    if "OutOfMemoryError" in output or "CUDA out of memory" in output:
        return "OOM"
    return f"ERROR(exit={returncode})"


def _apply_kernel_impl(row: dict, kernel_impl: dict | None) -> None:
    if not kernel_impl:
        return
    row["fla_version"] = kernel_impl.get("fla_version")
    row["triton_version"] = kernel_impl.get("triton_version")
    row["kernel_impl"] = kernel_impl.get("gated_delta_rule_impl")


def measure_infer(stage: str, long_side: int, n=4):
    set_stage(stage)
    row = {"stage": stage, "mode": "infer", "long_side": long_side, "crops": 0}
    try:
        rc, output = _run_subprocess([
            "smoke", "-c", "9b_base_1024", "--synthetic", "-n", str(n),
            "--set", f"input.long_side={long_side}",
        ])
        status = _classify_status(rc, output)
        summary_path = REPO_ROOT / "runs" / "smoke_9b_base_1024" / "summary.json"
        if status == "ok" and summary_path.exists():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            row["visual_tokens"] = summary.get("visual_tokens")
            row["peak_vram_torch"] = summary.get("peak_vram_gb")
            _apply_kernel_impl(row, summary.get("kernel_impl"))
        row["status"] = status
        print(f"  {stage}: {status}  peak_vram_torch={row.get('peak_vram_torch')}")
        if status != "ok":
            print(output[-1500:])
    except subprocess.TimeoutExpired:
        row["status"] = "TIMEOUT"
        print(f"  {stage}: TIMEOUT")
    except Exception as exc:
        row["status"] = f"ERROR({type(exc).__name__})"
        print(f"  {stage}: ERROR — {exc}")
    _write_result_row(row)
    mark_progress(stage, "done")
    return row


if RESUME_FROM <= 2:
    mark_progress("2-infer", "start")
    measure_infer("2-infer-1024", long_side=1024)
    measure_infer("2-infer-1536", long_side=1536)
    measure_infer("2-infer-2048", long_side=2048)
    mark_progress("2-infer", "done")
else:
    print("RESUME_FROM 설정에 따라 Stage 2를 건너뛴다.")


## Stage 3 — 학습 probe (가장 무거움, 셧다운 가능성 가장 높음, 맨 마지막)

Full 1024, NF4, r=8, grad_checkpointing ON, 12 step. **측정에 쓴 PC의 두 셧다운
전부 이 단계(또는 이만큼 무거운 학습)에서 났다 — 여기서 재현될 가능성이 가장
높다.** 앞 3 step은 warmup(Triton 컴파일 포함, 첫 step은 1~2분 걸릴 수 있다 —
정상, 여기서는 Ctrl+C 누르지 말 것), 이후 9 step의 중앙값을 steady-state로 본다.


In [ ]:
def measure_train(stage: str, n_steps=12):
    set_stage(stage)
    row = {"stage": stage, "mode": "train", "long_side": 1024, "crops": 0}
    try:
        rc, output = _run_subprocess([
            "train", "-c", "9b_qlora_r8_1024", "--synthetic", "-n", str(n_steps),
        ], timeout_sec=1800)
        status = _classify_status(rc, output)
        summary_path = REPO_ROOT / "runs" / "trainprobe_9b_qlora_r8_1024" / "summary.json"
        if status == "ok" and summary_path.exists():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            probe = summary.get("train_probe", {})
            row["peak_vram_torch"] = probe.get("peak_vram_gb_torch")
            row["peak_vram_smi"] = probe.get("peak_vram_gb_nvidia_smi")
            row["shared_mem_peak"] = probe.get("peak_shared_mem_gb_windows")
            row["sec_per_step_steady"] = probe.get("steady_state_sec_per_step_median")
            row["sec_per_step_warmup"] = probe.get("warmup_sec_per_step")
            _apply_kernel_impl(row, probe.get("kernel_impl"))
        row["status"] = status
        print(f"  {stage}: {status}  peak_vram_torch={row.get('peak_vram_torch')}  "
              f"steady sec/step={row.get('sec_per_step_steady')}  "
              f"shared_mem_peak={row.get('shared_mem_peak')}")
        if status != "ok":
            print(output[-1500:])
    except subprocess.TimeoutExpired:
        row["status"] = "TIMEOUT"
        print(f"  {stage}: TIMEOUT")
    except Exception as exc:
        row["status"] = f"ERROR({type(exc).__name__})"
        print(f"  {stage}: ERROR — {exc}")
    _write_result_row(row)
    mark_progress(stage, "done")
    return row


if RESUME_FROM <= 3:
    mark_progress("3-train", "start")
    measure_train("3-train-1024-r8-gradckpt-on")
    mark_progress("3-train", "done")
else:
    print("RESUME_FROM 설정에 따라 Stage 3를 건너뛴다.")


## 마무리

로거를 멈추고 파일 위치를 다시 출력한다. **셧다운으로 여기까지 못 왔어도
`gpulog_*.csv`와 `progress_*.json`은 이미 디스크에 있다** — 그 파일들만 챙겨서
보내면 된다.


In [ ]:
_gpulog_stop.set()
_gpulog_thread.join(timeout=5)
print("GPU 로거 정지.")
print("\n결과 파일 위치:")
print(" ", Path(RESULTS_PATH).resolve())
print(" ", Path(GPULOG_PATH).resolve())
print(" ", Path(PROGRESS_PATH).resolve())
print("\n이 세 파일을 보내주면 된다. 중간에 멈췄어도 gpulog/progress 파일은 그 직전까지 남아 있다.")
